# Overview

This notebook explores scraping **Apple Podcasts** (`podcasts.apple.com`): resolving visible **See All (number)** on the show page, opening the full episode list, scrolling until enough rows are in the DOM, and exporting deduplicated episode URLs.

It mirrors the logic in `scrape_podcasts.py` for interactive experimentation.

## Setup

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [2]:
EPISODE_LINK_SELECTOR = (By.CSS_SELECTOR, 'a[href*="/podcast/"][href*="?i="]')
SEE_ALL_REGEX = re.compile(r"See All\s*\((\d+)\)", re.IGNORECASE)
SCROLL_PAUSE_SEC = 1.2
MAX_SCROLL_WITHOUT_GROWTH = 6

podcast_url = (
    "https://podcasts.apple.com/us/podcast/more-attention-less-deficit/id312831485"
)
export_dir = Path("export")
export_dir.mkdir(parents=True, exist_ok=True)

In [3]:


def _unique_episode_hrefs(driver):
    hrefs = set()
    for el in driver.find_elements(*EPISODE_LINK_SELECTOR):
        href = el.get_attribute("href")
        if href and "podcasts.apple.com" in href:
            hrefs.add(href)
    return hrefs


def _locate_see_all(driver):
    candidates = driver.find_elements(
        By.XPATH,
        (
            "//a[contains(., 'See All')]"
            "|//*[@role='link'][contains(., 'See All')]"
            "|//button[contains(., 'See All')]"
        ),
    )
    for el in candidates:
        text = (el.text or el.get_attribute("textContent") or "").strip()
        m = SEE_ALL_REGEX.search(text)
        if m:
            return el, int(m.group(1))
    return None, None


def get_expected_episode_count(driver, wait):
    try:
        wait.until(lambda d: _locate_see_all(d)[0] is not None)
    except TimeoutException:
        raise RuntimeError(
            "Timed out waiting for a 'See All (number)' control on the show page."
        ) from None
    _, count = _locate_see_all(driver)
    return count


def click_see_all(driver, wait):
    el, _ = _locate_see_all(driver)
    if el is None:
        raise RuntimeError("Could not find a clickable 'See All (number)' control.")

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
    time.sleep(0.3)
    try:
        wait.until(EC.element_to_be_clickable(el))
    except TimeoutException:
        pass
    driver.execute_script("arguments[0].click();", el)


def scroll_until_all_loaded(driver, wait, expected_count):
    idle_rounds = 0

    while idle_rounds < MAX_SCROLL_WITHOUT_GROWTH:
        current = len(_unique_episode_hrefs(driver))
        if current >= expected_count:
            break

        before = current
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE_SEC)
        try:
            wait.until(
                lambda d: len(_unique_episode_hrefs(d)) > before
                or len(_unique_episode_hrefs(d)) >= expected_count
            )
        except TimeoutException:
            pass
        after = len(_unique_episode_hrefs(driver))

        if after <= before:
            idle_rounds += 1
        else:
            idle_rounds = 0


def get_episode_links(driver, wait):
    try:
        wait.until(EC.presence_of_element_located(EPISODE_LINK_SELECTOR))
    except TimeoutException:
        return []

    return sorted(_unique_episode_hrefs(driver))


options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

## Open main show page

In [4]:
driver.get(podcast_url)
print("Opened:", podcast_url)

Opened: https://podcasts.apple.com/us/podcast/more-attention-less-deficit/id312831485


## Find episode count from "See All (number)"

In [5]:
expected_count = get_expected_episode_count(driver, wait)
print("Expected episode count (from See All):", expected_count)

Expected episode count (from See All): 111


## Click "See All"

In [6]:
click_see_all(driver, wait)
print("Clicked See All; waiting for episode list…")

time.sleep(1.0)
try:
    wait.until(EC.presence_of_element_located(EPISODE_LINK_SELECTOR))
except TimeoutException:
    pass

Clicked See All; waiting for episode list…


## Scroll until all episodes are loaded

In [7]:
scroll_until_all_loaded(driver, wait, expected_count)
loaded = len(_unique_episode_hrefs(driver))
print("Unique episode links in DOM after scrolling:", loaded)

Unique episode links in DOM after scrolling: 25


## Extract episode links

In [8]:
episode_urls = get_episode_links(driver, wait)
print("Collected unique episode URLs:", len(episode_urls))
print("Preview:", episode_urls[:5])
episode_urls[:5]

Collected unique episode URLs: 25
Preview: ['https://podcasts.apple.com/us/podcast/a-guy-walks-into-a-chadd-conference-and-so-should-you/id312831485?i=1000120817792', 'https://podcasts.apple.com/us/podcast/adhd-is-depressing-but-it-doesnt-have-to-be/id312831485?i=1000092235565', 'https://podcasts.apple.com/us/podcast/awareness-honesty-and-willingness-the-three-keys/id312831485?i=1000160761940', 'https://podcasts.apple.com/us/podcast/be-willing-to-let-it-be-a-process/id312831485?i=1000094025333', 'https://podcasts.apple.com/us/podcast/change-more-pieces-to-turn-the-momentum/id312831485?i=1000104324321']


['https://podcasts.apple.com/us/podcast/a-guy-walks-into-a-chadd-conference-and-so-should-you/id312831485?i=1000120817792',
 'https://podcasts.apple.com/us/podcast/adhd-is-depressing-but-it-doesnt-have-to-be/id312831485?i=1000092235565',
 'https://podcasts.apple.com/us/podcast/awareness-honesty-and-willingness-the-three-keys/id312831485?i=1000160761940',
 'https://podcasts.apple.com/us/podcast/be-willing-to-let-it-be-a-process/id312831485?i=1000094025333',
 'https://podcasts.apple.com/us/podcast/change-more-pieces-to-turn-the-momentum/id312831485?i=1000104324321']

## Save results

In [9]:
output_path = export_dir / "episodes_all.csv"
pd.DataFrame({"episode_url": episode_urls}).to_csv(output_path, index=False)
print(f"Saved {output_path}")

Saved export/episodes_all.csv


## Notes

- **See All (N)** is parsed with a regex; candidate elements are `a`, `role=link`, or `button` so small DOM changes are less likely to break the notebook immediately.
- **Scrolling** stops when the unique link count reaches `N` or when the count stops growing for several iterations (avoids infinite loops).
- **`episode_urls`** is deduplicated and sorted for stable CSV output.

In [10]:
driver.quit()
print("Driver closed.")

Driver closed.


The main reusable version of this project lives in `scrape_podcasts.py`.
This notebook is kept only as a lightweight exploration record.